# IMDB Movie Sentiment Analysis
Using RNNs to classify sentiment on IMDB data

## 1. Importing relevant modules

In [32]:
import pandas as pd
import pickle
from zipfile import ZipFile

from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, save_model
from tensorflow.keras.layers import LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


## 2. Data collecction

Thee code below is loading the IMDB movie review dataset, but it limits the vocabulary size to 5000.

In [2]:
# Download dataset to the current directory
!kaggle datasets download -d lakshmi25npathi/imdb-dataset-of-50k-movie-reviews --path .


Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
imdb-dataset-of-50k-movie-reviews.zip: Skipping, found more recently modified local copy (use --force to force download)


In [3]:
# Unzip the dataset file
with ZipFile("imdb-dataset-of-50k-movie-reviews.zip", "r") as zip_ref:
    zip_ref.extractall()
    

## 3. Data exploration and cleaning

In [4]:
df = pd.read_csv("IMDB Dataset.csv")

In [5]:
df.shape

(50000, 2)

In [6]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [7]:
df.tail()

,review,sentiment
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative
49999,No one expects the Star Trek movies to be high...,negative


In [8]:
df["sentiment"].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [9]:
sentiment_mapping = {
    "sentiment":{"positive": 1, "negative": 0}
}
df = df.replace(sentiment_mapping).infer_objects(copy=False)

C:\Users\joeln\AppData\Local\Temp\ipykernel_24180\3988269745.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(sentiment_mapping).infer_objects(copy=False)


In [10]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [11]:
df["sentiment"].value_counts()

sentiment
1    25000
0    25000
Name: count, dtype: int64

## 4. Split data to training and test sets

In [12]:
train_data, test_data = train_test_split(df, test_size=0.2, random_state=42)

In [13]:
print(train_data.shape, test_data.shape)

(40000, 2) (10000, 2)


## 5. Data preprocessing

In [14]:
# Tokenize text data
tokenizer = Tokenizer(num_words=5000) # take most common 5,000 words as our vocabulary and converted to integers
tokenizer.fit_on_texts(train_data["review"]) # fit only on training data so as to avoid data leakage

# Convert data and apply padding
# Applying padding allows for all input texts to be of the same length, applying under padding or truncating
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200) # max length means total input will be 200, max 200 words
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200)


In [15]:
print(X_train.shape)
print(X_train)

(40000, 200)
[[1935    1 1200 ...  205  351 3856]
 [   3 1651  595 ...   89  103    9]
 [   0    0    0 ...    2  710   62]
 ...
 [   0    0    0 ... 1641    2  603]
 [   0    0    0 ...  245  103  125]
 [   0    0    0 ...   70   73 2062]]


In [16]:
print(X_test.shape)
print(X_test)

(10000, 200)
[[   0    0    0 ...  995  719  155]
 [  12  162   59 ...  380    7    7]
 [   0    0    0 ...   50 1088   96]
 ...
 [   0    0    0 ...  125  200 3241]
 [   0    0    0 ... 1066    1 2305]
 [   0    0    0 ...    1  332   27]]


In [17]:
y_train = train_data["sentiment"]
y_test = test_data['sentiment']

In [18]:
print(y_train.shape)
print(y_train)

(40000,)
39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64


In [19]:
print(y_test.shape)
print(y_test)

(10000,)
33553    1
9427     1
199      0
12447    1
39489    0
        ..
28567    0
25079    1
18707    1
15200    0
5857     1
Name: sentiment, Length: 10000, dtype: int64


## 6. Model Building
- LSTMs (Long Short-Term Memory networks) are highly effective for sentiment analysis because they excel at capturing long-term dependencies in text data.
- Unlike traditional neural networks or basic RNNs, LSTMs have gated mechanisms that help retain important contextual information while avoiding the vanishing gradient problem.
- This makes them ideal for understanding the sentiment of a sentence by considering the entire sequence rather than just isolated words.
- LSTMs are particularly useful when analyzing long reviews, social media posts, or sequential text, where context plays a crucial role in determining sentiment. 

In [20]:
# Create a sequential model
model = Sequential()

# Embedding layer:
# - input_dim=5000 → The vocabulary size (only the top 5000 most frequent words are considered).
# - output_dim=128 → Each word will be represented by a 128-dimensional dense vector.
# - input_length=200 → Each input sequence (review) will have a fixed length of 200 words.
model.add(Embedding(input_dim=5000, output_dim=128, input_length=200))

# LSTM (Long Short-Term Memory) layer:
# - 128 → Number of LSTM units (hidden neurons).
# - dropout=0.2 → Drops 20% of the input units to prevent overfitting.
# - recurrent_dropout=0.2 → Drops 20% of the recurrent connections to prevent overfitting.
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))

# Dense (Fully Connected) layer:
# - 1 → Output layer with a single neuron (for binary classification: positive/negative sentiment).
# - activation="sigmoid" → Sigmoid activation function outputs a probability (0 to 1), making it suitable for binary classification.
model.add(Dense(1, activation="sigmoid"))


c:\Users\joeln\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [21]:
# Compile the model before training  
model.compile(
    optimizer="adam",  # Adam optimizer (Adaptive Moment Estimation): adaptive learning rate optimization for better performance
    loss="binary_crossentropy",  # Loss function: suitable for binary classification (positive/negative sentiment)
    metrics=["accuracy"]  # Track accuracy during training and evaluation
)


In [22]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## 7. Model Training

In [23]:
# Train the model on the training dataset
model.fit(
    X_train,  # Input training data (features)
    y_train,  # Target labels (0 for negative, 1 for positive)
    epochs=5,  # Number of times the model sees the full dataset
    batch_size=64,  # Number of samples processed before updating model weights
    validation_split=0.2  # Use 20% of training data for validation
)


Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 138s 273ms/step - accuracy: 0.7302 - loss: 0.5278 - val_accuracy: 0.8295 - val_loss: 0.3900
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 63s 126ms/step - accuracy: 0.8557 - loss: 0.3552 - val_accuracy: 0.8444 - val_loss: 0.3570
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 62s 125ms/step - accuracy: 0.8792 - loss: 0.3009 - val_accuracy: 0.8430 - val_loss: 0.3701
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 63s 127ms/step - accuracy: 0.8782 - loss: 0.2993 - val_accuracy: 0.8599 - val_loss: 0.3261
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 63s 126ms/step - accuracy: 0.9070 - loss: 0.2365 - val_accuracy: 0.8734 - val_loss: 0.3313


In [24]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ (64, 200, 128)         │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (64, 128)              │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (64, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,315,141 (8.83 MB)

 Trainable params: 771,713 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,543,428 (5.89 MB)

# 8. Model Evaluation

In [25]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.8804 - loss: 0.3216
Test Loss: 0.30881258845329285
Test Accuracy: 0.8851000070571899


# 9. Build Predictive System

In [26]:
def predict_sentiment(review):
    # Convert the input review text into a sequence of word indices
    sequence = tokenizer.texts_to_sequences([review])
    
    # Pad the sequence to ensure it has a fixed length of 200 words
    # If the review is shorter than 200 words, it will be padded with zeros
    # If it's longer, it will be truncated from the beginning
    padded_sequence = pad_sequences(sequence, maxlen=200)
    
    # Use the trained model to predict the sentiment of the review
    prediction = model.predict(padded_sequence)
    print(prediction)
    
    # If the predicted value is greater than 0.5, classify it as "positive"
    # Otherwise, classify it as "negative"
    sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
    
    return sentiment  # Return the predicted sentiment


In [29]:
# Example usage
new_review = "This movie was fantastic. I loved it."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is {sentiment}.")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
[[0.7721989]]
The sentiment of the review is positive.


In [30]:
# Example usage
new_review = "This movie not that good."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is {sentiment}.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
[[0.36030662]]
The sentiment of the review is negative.


In [31]:
# Example usage
new_review = "This movie was ok but not that good."
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is {sentiment}.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
[[0.2745397]]
The sentiment of the review is negative.


# 10. Save model and tokenizer

In [34]:
# Save the model and tokenizer
save_model(model, 'sentiment_model.keras')
with open('tokenizer.pickle', 'wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
